<a href="https://colab.research.google.com/github/MeouchyCharbel/Neural-Machine-Translation/blob/feature%2Fevaluation-improvements/02_evaluation_improvements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip install -q transformers torch sentencepiece sacremoses sacrebleu pandas scikit-learn matplotlib


In [2]:
import glob
import urllib.request
import zipfile

DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip"
ZIP_PATH = "fra-eng.zip"
EXTRACT_DIR = "data_raw"

urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as zip_file:
    zip_file.extractall(EXTRACT_DIR)

text_file = glob.glob(f"{EXTRACT_DIR}/**/fra.txt", recursive=True)[0]

print("Dataset file:", text_file)

Dataset file: data_raw/fra.txt


In [3]:
import csv
import pandas as pd

pairs = pd.read_csv(
    text_file,
    sep="\t",
    header=None,
    usecols=[0, 1],
    names=["en", "fr"],
    quoting=csv.QUOTE_NONE,
    encoding="utf-8"
)

print("Number of pairs:", len(pairs))
pairs.sample(5, random_state=42)

Number of pairs: 167130


,en,fr
59241,I think it's fascinating.,Je pense que c'est fascinant.
135841,Why don't we talk about something else?,Pourquoi ne parlons-nous pas d'autre chose ?
113994,I don't really know what you mean.,Je ne sais pas vraiment ce que tu veux dire.
138328,Those prisoners were set free yesterday.,Ces prisonnières ont été libérées hier.
31886,You're a funny girl.,T'es une drôle de fille.


In [4]:
rows_before = len(pairs)

pairs = pairs.dropna(subset=["en", "fr"]).copy()

pairs["en"] = pairs["en"].str.strip()
pairs["fr"] = pairs["fr"].str.strip()

pairs["en_words"] = pairs["en"].str.split().str.len()

pairs = pairs[(pairs["en"] != "") & (pairs["fr"] != "")]
pairs = pairs.drop_duplicates(subset=["en", "fr"])
pairs = pairs[pairs["en_words"] <= 15]

pairs = pairs.reset_index(drop=True)

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", len(pairs))
print("Rows removed:", rows_before - len(pairs))
print("Unique English sentences:", pairs["en"].nunique())
print("Longest English sentence:", pairs["en_words"].max(), "words")

Rows before cleaning: 167130
Rows after cleaning: 166340
Rows removed: 790
Unique English sentences: 116198
Longest English sentence: 15 words


In [5]:

import re
import unicodedata
from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def loose_normalize(text):
    text = normalize_text(text)
    text = re.sub(r"[\W_]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

pairs["en_key"] = pairs["en"].apply(normalize_text)
pairs["fr_key"] = pairs["fr"].apply(normalize_text)
pairs["en_loose"] = pairs["en"].apply(loose_normalize)
pairs["fr_loose"] = pairs["fr"].apply(loose_normalize)

parent = list(range(len(pairs)))
size = [1] * len(pairs)

def find(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index

def union(a, b):
    root_a = find(a)
    root_b = find(b)

    if root_a == root_b:
        return

    if size[root_a] < size[root_b]:
        root_a, root_b = root_b, root_a

    parent[root_b] = root_a
    size[root_a] += size[root_b]

seen = {}

for i, (english, french) in enumerate(
    zip(pairs["en_loose"], pairs["fr_loose"])
):
    for key in [("en", english), ("fr", french)]:
        if key in seen:
            union(i, seen[key])
        else:
            seen[key] = i

pairs["group_id"] = [
    find(i) for i in range(len(pairs))
]

first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, remaining_idx = next(
    first_split.split(pairs, groups=pairs["group_id"])
)

train_pairs = pairs.iloc[train_idx].reset_index(drop=True)
remaining_pairs = pairs.iloc[remaining_idx].reset_index(drop=True)

second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=RANDOM_STATE
)

val_idx, test_idx = next(
    second_split.split(
        remaining_pairs,
        groups=remaining_pairs["group_id"]
    )
)

val_pairs = remaining_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = remaining_pairs.iloc[test_idx].reset_index(drop=True)

splits = {
    "Train": train_pairs,
    "Validation": val_pairs,
    "Test": test_pairs
}

columns_to_check = [
    "en_key",
    "fr_key",
    "en_loose",
    "fr_loose",
    "group_id"
]

total = len(pairs)

print("DATASET SPLIT SUMMARY")

for name, df in splits.items():
    print(
        f"{name}: {len(df)} pairs "
        f"({len(df) / total:.2%})"
    )

print("\nNUMBER OF CONNECTED GROUPS")
print(pairs["group_id"].nunique())

print("\nLARGEST GROUP SIZE")
print(pairs["group_id"].value_counts().max())

print("\nOVERLAP VERIFICATION")

comparisons = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

for left_name, right_name in comparisons:
    print(f"\n{left_name} vs {right_name}")

    for column in columns_to_check:
        overlap = len(
            set(splits[left_name][column])
            & set(splits[right_name][column])
        )

        print(f"{column}: {overlap}")

        assert overlap == 0

assert sum(len(df) for df in splits.values()) == total

print("\nAll overlap checks passed.")
print("Every dataset row belongs to exactly one split.")


DATASET SPLIT SUMMARY
Train: 133017 pairs (79.97%)
Validation: 16616 pairs (9.99%)
Test: 16707 pairs (10.04%)

NUMBER OF CONNECTED GROUPS
108720

LARGEST GROUP SIZE
96

OVERLAP VERIFICATION

Train vs Validation
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0
group_id: 0

Train vs Test
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0
group_id: 0

Validation vs Test
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0
group_id: 0

All overlap checks passed.
Every dataset row belongs to exactly one split.


In [6]:

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def loose_normalize(text):
    text = normalize_text(text)
    text = re.sub(r"[\W_]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

splits = {
    "Train": train_pairs,
    "Validation": val_pairs,
    "Test": test_pairs
}

for name, df in splits.items():
    df["fr_key"] = df["fr"].apply(normalize_text)
    df["en_loose"] = df["en"].apply(loose_normalize)
    df["fr_loose"] = df["fr"].apply(loose_normalize)

def count_overlap(left, right, column):
    return len(set(left[column]) & set(right[column]))

comparisons = [
    ("Train", "Validation"),
    ("Train", "Test"),
    ("Validation", "Test")
]

print("EXACT AND PUNCTUATION-INSENSITIVE OVERLAP")

for left_name, right_name in comparisons:
    left = splits[left_name]
    right = splits[right_name]

    print(f"\n{left_name} vs {right_name}")

    for column in ["en_key", "fr_key", "en_loose", "fr_loose"]:
        count = count_overlap(left, right, column)
        print(f"{column}: {count}")

train_sample = train_pairs.drop_duplicates("en_key").sample(
    n=min(12000, train_pairs["en_key"].nunique()),
    random_state=42
).reset_index(drop=True)

validation_sample = val_pairs.drop_duplicates("en_key").sample(
    n=min(150, val_pairs["en_key"].nunique()),
    random_state=42
)

test_sample = test_pairs.drop_duplicates("en_key").sample(
    n=min(150, test_pairs["en_key"].nunique()),
    random_state=42
)

queries = pd.concat(
    [
        validation_sample.assign(split="Validation"),
        test_sample.assign(split="Test")
    ],
    ignore_index=True
)

vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    lowercase=True
)

all_sentences = (
    train_sample["en"].tolist()
    + queries["en"].tolist()
)

vectors = vectorizer.fit_transform(all_sentences)

train_vectors = vectors[:len(train_sample)]
query_vectors = vectors[len(train_sample):]

matches = []

for start in range(0, len(queries), 16):
    end = min(start + 16, len(queries))

    scores = cosine_similarity(
        query_vectors[start:end],
        train_vectors
    )

    best_indices = np.argmax(scores, axis=1)
    best_scores = np.max(scores, axis=1)

    for offset, (index, score) in enumerate(
        zip(best_indices, best_scores)
    ):
        query = queries.iloc[start + offset]
        training = train_sample.iloc[index]

        if score >= 0.85:
            matches.append({
                "split": query["split"],
                "sentence": query["en"],
                "similar_training_sentence": training["en"],
                "similarity": round(float(score), 4)
            })

matches_df = pd.DataFrame(
    matches,
    columns=[
        "split",
        "sentence",
        "similar_training_sentence",
        "similarity"
    ]
)

print("\nSAMPLED NEAR-DUPLICATE SCREEN")
print("Training sentences sampled:", len(train_sample))
print("Validation/test sentences sampled:", len(queries))
print("Pairs with similarity >= 0.85:", len(matches_df))

display(
    matches_df.sort_values(
        "similarity",
        ascending=False
    ).head(15)
)


EXACT AND PUNCTUATION-INSENSITIVE OVERLAP

Train vs Validation
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0

Train vs Test
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0

Validation vs Test
en_key: 0
fr_key: 0
en_loose: 0
fr_loose: 0

SAMPLED NEAR-DUPLICATE SCREEN
Training sentences sampled: 12000
Validation/test sentences sampled: 300
Pairs with similarity >= 0.85: 2


,split,sentence,similar_training_sentence,similarity
0,Validation,Look up the words in your dictionary.,Look up the new words in your dictionary.,0.8901
1,Validation,I think that's enough for now.,That's enough for now.,0.8897


In [7]:

import re

dictionary = {
    "i": "je",
    "am": "suis",
    "happy": "heureux",
    "she": "elle",
    "reads": "lit",
    "a": "un",
    "book": "livre",
    "where": "où",
    "is": "est",
    "the": "la",
    "station": "gare"
}

def baseline_translate(sentence):
    words = re.findall(r"\w+|[^\w\s]", sentence.lower())

    translated_words = [
        dictionary.get(word, word)
        for word in words
    ]

    translation = " ".join(translated_words)
    translation = re.sub(r"\s+([.,!?])", r"\1", translation)

    return translation.capitalize()

print(baseline_translate("She reads a book."))


Elle lit un livre.


In [8]:

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

retrieval_data = train_pairs.drop_duplicates(
    subset="en_key"
).reset_index(drop=True)

retrieval_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
    dtype=np.float32
)

retrieval_matrix = retrieval_vectorizer.fit_transform(
    retrieval_data["en"]
)

def retrieval_translate(sentences, batch_size=16):
    predictions = []
    matched_sentences = []
    similarity_scores = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        query_matrix = retrieval_vectorizer.transform(batch)

        similarities = linear_kernel(
            query_matrix,
            retrieval_matrix
        )

        best_indices = np.argmax(similarities, axis=1)
        best_scores = np.max(similarities, axis=1)

        predictions.extend(
            retrieval_data["fr"].iloc[best_indices].tolist()
        )

        matched_sentences.extend(
            retrieval_data["en"].iloc[best_indices].tolist()
        )

        similarity_scores.extend(best_scores.tolist())

    return predictions, matched_sentences, similarity_scores

example_sentences = [
    "I am very tired.",
    "Where is the train station?",
    "She reads a book."
]

predictions, matches, scores = retrieval_translate(
    example_sentences
)

for english, matched, french, score in zip(
    example_sentences,
    matches,
    predictions,
    scores
):
    print("Input:", english)
    print("Matched training sentence:", matched)
    print("French translation:", french)
    print("Similarity:", round(score, 4))
    print("-" * 60)

print("Training sentences indexed:", len(retrieval_data))
print("TF-IDF vocabulary size:", len(retrieval_vectorizer.vocabulary_))


Input: I am very tired.
Matched training sentence: I am very tired.
French translation: Je suis très fatigué.
Similarity: 1.0
------------------------------------------------------------
Input: Where is the train station?
Matched training sentence: Where is the train station?
French translation: Où est la gare ?
Similarity: 1.0
------------------------------------------------------------
Input: She reads a book.
Matched training sentence: She reads the newspaper every morning.
French translation: Elle lit le journal chaque matin.
Similarity: 0.5139
------------------------------------------------------------
Training sentences indexed: 92892
TF-IDF vocabulary size: 49304


In [9]:

import time

validation_df = val_pairs.drop_duplicates(
    subset="en_key"
).sample(
    n=min(500, val_pairs["en_key"].nunique()),
    random_state=42
).copy().reset_index(drop=True)

start = time.perf_counter()

validation_df["dictionary_translation"] = (
    validation_df["en"].apply(baseline_translate)
)

dictionary_time = time.perf_counter() - start

start = time.perf_counter()

retrieval_predictions, retrieval_matches, retrieval_scores = (
    retrieval_translate(validation_df["en"].tolist())
)

retrieval_time = time.perf_counter() - start

validation_df["retrieval_translation"] = retrieval_predictions
validation_df["retrieval_match"] = retrieval_matches
validation_df["retrieval_similarity"] = retrieval_scores

print("Number of validation sentences:", len(validation_df))

print("\nDICTIONARY BASELINE")
print("Total time:", round(dictionary_time, 4), "seconds")
print(
    "Average time per sentence:",
    round(dictionary_time / len(validation_df) * 1000, 4),
    "ms"
)

print("\nTF-IDF RETRIEVAL BASELINE")
print("Total time:", round(retrieval_time, 4), "seconds")
print(
    "Average time per sentence:",
    round(retrieval_time / len(validation_df) * 1000, 4),
    "ms"
)

display(
    validation_df[
        [
            "en",
            "fr",
            "dictionary_translation",
            "retrieval_translation",
            "retrieval_similarity"
        ]
    ].head(10)
)


Number of validation sentences: 500

DICTIONARY BASELINE
Total time: 0.0092 seconds
Average time per sentence: 0.0185 ms

TF-IDF RETRIEVAL BASELINE
Total time: 2.4917 seconds
Average time per sentence: 4.9834 ms


,en,fr,dictionary_translation,retrieval_translation,retrieval_similarity
0,Look up the words in your dictionary.,Cherchez les mots dans votre dictionnaire.,Look up la words in your dictionary.,Cherchez les nouveaux mots dans votre dictionn...,0.858194
1,I arrived at the station ten minutes late.,Je suis arrivé à la gare dix minutes en retard.,Je arrived at la gare ten minutes late.,Je suis arrivé à la gare.,0.700920
2,I don't have the authority to order that.,Je ne dispose pas de l'autorité nécessaire pou...,Je don ' t have la authority to order that.,Je ne dispose pas de l'autorité pour faire ça.,0.735560
3,We exchanged phone numbers at the end of the g...,Nous avons échangé nos numéros de téléphone à ...,We exchanged phone numbers at la end of la gat...,"Pour moi, c'est la fin des haricots.",0.528533
4,"If that guitar were not so expensive, I could ...","Si cette guitare n'était pas si chère, je pour...","If that guitar were not so expensive, je could...","Si cette guitare n'était pas aussi chère, je p...",0.854341
5,Why are you leaving so soon?,Pourquoi partez-vous si tôt ?,Why are you leaving so soon?,Pourquoi jures-tu ?,0.593796
6,You have to go.,Il faut que tu t'en ailles.,You have to go.,Je dois y aller.,0.855595
7,I needed time to convince her.,Il m'a fallu du temps pour la convaincre.,Je needed time to convince her.,J'ai eu du mal à la convaincre.,0.481714
8,I made sure of that.,Je m'en suis assuré.,Je made sure of that.,Nous en sommes tous certains.,0.573291
9,Where did you guys grow up?,"Où avez-vous grandi, les mecs ?",Où did you guys grow up?,"Où êtes-vous allés, les mecs ?",0.759147


In [10]:


import sacrebleu
import pandas as pd

references = validation_df["fr"].tolist()

methods = {
    "Dictionary Baseline": {
        "predictions": validation_df["dictionary_translation"].tolist(),
        "time": dictionary_time
    },
    "TF-IDF Retrieval": {
        "predictions": validation_df["retrieval_translation"].tolist(),
        "time": retrieval_time
    }
}

metrics_rows = []

for name, method in methods.items():
    predictions = method["predictions"]

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score

    metrics_rows.append({
        "Model": name,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2),
        "Total Time (s)": round(method["time"], 4),
        "Average Time (ms)": round(
            method["time"] / len(predictions) * 1000,
            4
        )
    })

baseline_metrics_df = pd.DataFrame(metrics_rows)

print("VALIDATION BASELINE COMPARISON")
print("Evaluation sentences:", len(references))

display(baseline_metrics_df)


VALIDATION BASELINE COMPARISON
Evaluation sentences: 500


,Model,BLEU,chrF,Total Time (s),Average Time (ms)
0,Dictionary Baseline,0.66,16.14,0.0092,0.0185
1,TF-IDF Retrieval,18.39,37.74,2.4917,4.9834


In [11]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model.eval()

print("Model loaded successfully!")
print("Device:", device)


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  301MB            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  301MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model loaded successfully!
Device: cpu


In [12]:

def transformer_translate(sentences, num_beams=1):
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    ).to(device)

    generation_settings = {
        "max_new_tokens": 64,
        "num_beams": num_beams,
        "do_sample": False
    }

    if num_beams > 1:
        generation_settings["early_stopping"] = True

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **generation_settings
        )

    translations = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )

    return translations


sample_df = validation_df.head(3)

configurations = {
    "Greedy": 1,
    "Beam Search 3": 3,
    "Beam Search 5": 5
}

for name, beams in configurations.items():
    predictions = transformer_translate(
        sample_df["en"].tolist(),
        num_beams=beams
    )

    print("\n", name)
    print("-" * 60)

    for english, reference, prediction in zip(
        sample_df["en"],
        sample_df["fr"],
        predictions
    ):
        print("English:", english)
        print("Reference:", reference)
        print("Translation:", prediction)
        print()


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Greedy
------------------------------------------------------------
English: Look up the words in your dictionary.
Reference: Cherchez les mots dans votre dictionnaire.
Translation: Regarde les mots de ton dictionnaire.

English: I arrived at the station ten minutes late.
Reference: Je suis arrivé à la gare dix minutes en retard.
Translation: Je suis arrivé à la gare dix minutes en retard.

English: I don't have the authority to order that.
Reference: Je ne dispose pas de l'autorité nécessaire pour passer commande de ça.
Translation: Je n'ai pas le pouvoir d'ordonner ça.



[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Beam Search 3
------------------------------------------------------------
English: Look up the words in your dictionary.
Reference: Cherchez les mots dans votre dictionnaire.
Translation: Regarde les mots de ton dictionnaire.

English: I arrived at the station ten minutes late.
Reference: Je suis arrivé à la gare dix minutes en retard.
Translation: Je suis arrivé à la gare dix minutes en retard.

English: I don't have the authority to order that.
Reference: Je ne dispose pas de l'autorité nécessaire pour passer commande de ça.
Translation: Je n'ai pas le pouvoir d'ordonner ça.


 Beam Search 5
------------------------------------------------------------
English: Look up the words in your dictionary.
Reference: Cherchez les mots dans votre dictionnaire.
Translation: Regarde les mots de ton dictionnaire.

English: I arrived at the station ten minutes late.
Reference: Je suis arrivé à la gare dix minutes en retard.
Translation: Je suis arrivé à la gare dix minutes en retard.

English: 

In [13]:

import time
import torch
import sacrebleu
import pandas as pd

sentences = validation_df["en"].tolist()
references = validation_df["fr"].tolist()

batch_size = 8

configurations = {
    "Greedy": 1,
    "Beam Search 3": 3,
    "Beam Search 5": 5
}

def translate_batches(sentences, num_beams, batch_size=8):
    predictions = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        translations = transformer_translate(
            batch,
            num_beams=num_beams
        )

        predictions.extend(translations)

    return predictions

print("Warming up the model...")

for name, beams in configurations.items():
    translate_batches(
        sentences[:8],
        num_beams=beams,
        batch_size=batch_size
    )

results = []

for name, beams in configurations.items():
    print(f"\nEvaluating: {name}")

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    predictions = translate_batches(
        sentences,
        num_beams=beams,
        batch_size=batch_size
    )

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    bleu = sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score

    chrf = sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score

    column_name = (
        "transformer_" +
        name.lower().replace(" ", "_")
    )

    validation_df[column_name] = predictions

    results.append({
        "Configuration": name,
        "BLEU": round(bleu, 2),
        "chrF": round(chrf, 2),
        "Total Time (s)": round(elapsed_time, 2),
        "Average Time (ms)": round(
            elapsed_time / len(sentences) * 1000,
            2
        ),
        "Sentences/Second": round(
            len(sentences) / elapsed_time,
            2
        )
    })

decoding_results_df = pd.DataFrame(results)

print("\nTRANSFORMER DECODING COMPARISON")
print("Validation sentences:", len(sentences))
print("Batch size:", batch_size)
print("Device:", device)

display(decoding_results_df)

print("\nTRANSLATION DIFFERENCES")

greedy = validation_df["transformer_greedy"]

for name in ["beam_search_3", "beam_search_5"]:
    other = validation_df[f"transformer_{name}"]

    different = (greedy != other).sum()

    print(
        f"Greedy vs {name}: "
        f"{different}/{len(validation_df)} "
        "translations differ"
    )


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Warming up the model...


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluating: Greedy


[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


Evaluating: Beam Search 3


... (2 outputs left out)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


TRANSFORMER DECODING COMPARISON
Validation sentences: 500
Batch size: 8
Device: cpu


,Configuration,BLEU,chrF,Total Time (s),Average Time (ms),Sentences/Second
0,Greedy,55.11,70.40,76.34,152.67,6.55
1,Beam Search 3,55.19,70.39,123.32,246.64,4.05
2,Beam Search 5,54.87,70.12,214.10,428.21,2.34



TRANSLATION DIFFERENCES
Greedy vs beam_search_3: 99/500 translations differ
Greedy vs beam_search_5: 103/500 translations differ


In [14]:

import pandas as pd
import sacrebleu

review_df = validation_df[
    [
        "en",
        "fr",
        "transformer_greedy",
        "transformer_beam_search_3",
        "transformer_beam_search_5"
    ]
].copy()

review_df["greedy_chrf"] = [
    round(sacrebleu.sentence_chrf(pred, [ref]).score, 2)
    for pred, ref in zip(
        review_df["transformer_greedy"],
        review_df["fr"]
    )
]

review_df["decoding_disagreement"] = (
    (review_df["transformer_greedy"] != review_df["transformer_beam_search_3"])
    |
    (review_df["transformer_greedy"] != review_df["transformer_beam_search_5"])
)

worst_ids = review_df.nsmallest(
    15, "greedy_chrf"
).index.tolist()

disagreement_pool = review_df[
    review_df["decoding_disagreement"]
].drop(index=worst_ids, errors="ignore")

disagreement_ids = disagreement_pool.nsmallest(
    15, "greedy_chrf"
).index.tolist()

remaining = review_df.drop(
    index=worst_ids + disagreement_ids
)

random_ids = remaining.sample(
    n=min(10, len(remaining)),
    random_state=42
).index.tolist()

selected_ids = worst_ids + disagreement_ids + random_ids

error_review_df = review_df.loc[
    selected_ids
].copy()

error_review_df["selection_reason"] = (
    ["Low chrF"] * len(worst_ids)
    + ["Decoding disagreement"] * len(disagreement_ids)
    + ["Random sample"] * len(random_ids)
)

error_review_df["meaning_preserved"] = ""
error_review_df["error_categories"] = ""
error_review_df["review_notes"] = ""

error_review_df = error_review_df.reset_index(drop=True)

error_review_df.to_csv(
    "error_review_template.csv",
    index=False
)

print("SYSTEMATIC ERROR ANALYSIS")
print("Total examples:", len(error_review_df))

print("\nSampling categories:")
print(error_review_df["selection_reason"].value_counts())

print("\nLOWEST SCORING TRANSLATIONS")

display(
    error_review_df[
        [
            "en",
            "fr",
            "transformer_greedy",
            "greedy_chrf"
        ]
    ].head(10)
)

print("\nError review template created successfully.")


SYSTEMATIC ERROR ANALYSIS
Total examples: 40

Sampling categories:
selection_reason
Low chrF                 15
Decoding disagreement    15
Random sample            10
Name: count, dtype: int64

LOWEST SCORING TRANSLATIONS


,en,fr,transformer_greedy,greedy_chrf
0,Here we are.,On y est.,Nous y voilà.,6.41
1,You'll give me away.,Tu me trahiras.,Vous me donnerez.,8.81
2,I was discouraged.,On me dissuada.,J'étais découragé.,9.76
3,Is it important?,C'est grave ?,Est-ce important ?,9.85
4,Just take my word for it.,Il suffit de me croire sur parole.,Je vous le promets.,11.15
5,It's bedtime.,Il est temps de dormir.,C'est l'heure du coucher.,13.88
6,He died the next day.,Il mourut le jour suivant.,Il est mort le lendemain.,14.75
7,I like my steak medium.,"Un steak à point, s’il vous plaît.",J'aime mon médium à steak.,14.82
8,We're friends now.,Nous sommes désormais amis.,On est amis maintenant.,14.87
9,Don't stick your nose where it doesn't belong.,Ne fourre pas le nez dans un vase étranger.,Ne colle pas ton nez là où il n'est pas.,15.36



Error review template created successfully.
